# Voice verification

Optional voice setup uses the pinned pretrained ECAPA model. Use distinct and consented enrollment and probe recordings.

Run setup from a local checkout. Optional installation, downloads and training are off by default.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# In Colab, put an authorized copy of this repository in /content or in Drive.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path(os.environ.get("TOPO_LOCK_REPO", ".")).resolve()
if not (ROOT / "pyproject.toml").is_file() and (ROOT.parent / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Set TOPO_LOCK_REPO to the local Acoustic Topo-Lock checkout.")

def run(*args):
    subprocess.run([sys.executable, *args], cwd=ROOT, check=True)

# Set TOPO_LOCK_DATA_DIR and TOPO_LOCK_MODEL_DIR to mounted persistent storage if needed.
INSTALL = False
if INSTALL:
    run("-m", "pip", "install", "-e", ".[research]")

In [ ]:
from acoustic_topo_lock.application import AuthenticationService
from acoustic_topo_lock.core.config_loader import load_settings
from acoustic_topo_lock.core.status import Check

FETCH_WEIGHTS = False
if FETCH_WEIGHTS:
    run("-m", "pip", "install", "-r", "requirements-ml.txt")
    run("-m", "scripts.fetch_ecapa")

service = AuthenticationService(load_settings(root=ROOT))
record = service.registry.inspect(Check.VOICE)
print(record.status)
REFERENCE_PATHS = []
PROBE = None
CONSENT_CONFIRMED = False
if record.available and REFERENCE_PATHS and PROBE and CONSENT_CONFIRMED:
    profile = service.enroll_voice("participant", [(p, None) for p in REFERENCE_PATHS], consented=True)
    print(service.voice.verify(service.analyze(PROBE), profile))